In [8]:
import pandas as pd
import numpy as np
from scipy.optimize import least_squares
import matplotlib.pyplot as plt
from pathlib import Path

In [9]:
csv_path = "data/similarweb_genai_platform_visits_2025.csv"
df = pd.read_csv(csv_path)
df["month"] = pd.to_datetime(df["month"])

In [10]:
def bass_F(t: np.ndarray, p: float, q: float) -> np.ndarray:
    t = np.asarray(t, dtype=float)
    exp_term = np.exp(-(p + q) * t)
    return (1.0 - exp_term) / (1.0 + (q / p) * exp_term)

def bass_cumulative(t: np.ndarray, m: float, p: float, q: float) -> np.ndarray:
    return m * bass_F(t, p, q)

def bass_sales_by_diff(t: np.ndarray, m: float, p: float, q: float) -> np.ndarray:
    t = np.asarray(t, dtype=float)
    A_t = bass_cumulative(t, m, p, q)
    A_prev = bass_cumulative(np.maximum(t - 1.0, 0.0), m, p, q)
    return A_t - A_prev

In [11]:
def fit_bass_sales(t: np.ndarray, y: np.ndarray, m_fixed=None):
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)

    if m_fixed is None:
        m0 = max(y.sum() * 1.2, y.max() * 2.0)
        x0 = np.array([m0, 0.03, 0.4], dtype=float)
        lb = np.array([y.sum(), 1e-6, 1e-6], dtype=float)
        ub = np.array([y.sum() * 100.0, 5.0, 20.0], dtype=float)

        def resid(x):
            m, p, q = x
            return bass_sales_by_diff(t, m, p, q) - y

        sol = least_squares(resid, x0=x0, bounds=(lb, ub), loss="linear")
        m, p, q = sol.x
    else:
        m = float(m_fixed)
        x0 = np.array([0.03, 0.4], dtype=float)
        lb = np.array([1e-6, 1e-6], dtype=float)
        ub = np.array([5.0, 20.0], dtype=float)

        def resid(x):
            p, q = x
            return bass_sales_by_diff(t, m, p, q) - y

        sol = least_squares(resid, x0=x0, bounds=(lb, ub), loss="linear")
        p, q = sol.x

    yhat = bass_sales_by_diff(t, m, p, q)
    rmse = float(np.sqrt(np.mean((yhat - y) ** 2)))
    mape = float(np.mean(np.abs((yhat - y) / np.maximum(y, 1e-12))) * 100.0)
    return {"M": float(m), "p": float(p), "q": float(q), "RMSE": rmse, "MAPE_pct": mape}, yhat

def run_fit(platform: str, scale=1e6):
    y_raw = df[platform].astype(float).to_numpy()
    y = y_raw / scale
    t = np.arange(1, len(y) + 1, dtype=float)
    fit, yhat = fit_bass_sales(t, y, m_fixed=None)
    return fit, y, yhat, t

In [12]:
out_dir = Path("outputs")
out_dir.mkdir(parents=True, exist_ok=True)
scale = 1e6
forecast_months = 12
prefix = "bass2025"

In [13]:
def build_output(platform: str):
    fit, y, yhat, t = run_fit(platform, scale=scale)
    T = len(y)
    t_all = np.arange(1, T + forecast_months + 1, dtype=float)
    yhat_all = bass_sales_by_diff(t_all, fit["M"], fit["p"], fit["q"])
    Ahat_all = np.cumsum(yhat_all)
    months_all = pd.date_range(df["month"].iloc[0], periods=len(t_all), freq="MS")

    out = pd.DataFrame({
        "month": months_all,
        "observed_per_period": np.r_[y, [np.nan] * forecast_months],
        "fitted_per_period": yhat_all,
        "fitted_cumulative": Ahat_all,
    })

    out_csv = out_dir / f"{prefix}_{platform.replace('.','_')}.csv"
    out.to_csv(out_csv, index=False)

    fig = plt.figure()
    plt.plot(df["month"], y, marker="o", linewidth=1, label="Observed (scaled)")
    plt.plot(df["month"], yhat, marker="o", linewidth=1, label="Fitted (scaled)")
    plt.title(f"Bass fit: {platform} (RMSE={fit['RMSE']:.3f}, MAPE={fit['MAPE_pct']:.1f}%)")
    plt.xlabel("Month")
    plt.ylabel(f"Per-period proxy (visits / {scale:g})")
    plt.legend()
    fig.autofmt_xdate()
    fit_png = out_dir / f"{prefix}_{platform.replace('.','_')}_fit.png"
    fig.savefig(fit_png, dpi=160, bbox_inches="tight")
    plt.close(fig)

    fig2 = plt.figure()
    plt.plot(months_all, Ahat_all, marker="o", linewidth=1)
    plt.title(f"Fitted cumulative (and forecast): {platform}")
    plt.xlabel("Month")
    plt.ylabel(f"Cumulative (sum of visits / {scale:g})")
    fig2.autofmt_xdate()
    cum_png = out_dir / f"{prefix}_{platform.replace('.','_')}_cumulative.png"
    fig2.savefig(cum_png, dpi=160, bbox_inches="tight")
    plt.close(fig2)

    preview = out.head(15)
    preview_csv = out_dir / f"{prefix}_{platform.replace('.','_')}_preview.csv"
    preview.to_csv(preview_csv, index=False)

    return fit, out_csv, fit_png, cum_png, preview_csv

In [14]:
platforms = ["deepseek.com", "chatgpt.com"]
fit_rows = []
artifacts = {}

for p in platforms:
    fit, out_csv, fit_png, cum_png, preview_csv = build_output(p)
    fit_rows.append({"platform": p, **fit})
    artifacts[p] = {
        "out_csv": str(out_csv),
        "fit_png": str(fit_png),
        "cum_png": str(cum_png),
        "preview_csv": str(preview_csv),
    }

fit_df = pd.DataFrame(fit_rows)
param_csv = out_dir / f"{prefix}_parameters_scaled_millions.csv"
fit_df.to_csv(param_csv, index=False)

print(fit_df)
print("Saved:", str(param_csv))
for p in platforms:
    print(p, artifacts[p])

# Expected parameter values (millions scale) from my run:
# deepseek.com:  M=7287.400066, p=0.061430, q=0.074284, RMSE=77.687406, MAPE=15.429151
# chatgpt.com:   M=111177.737307, p=0.031950, q=0.144526, RMSE=149.131812, MAPE=2.504960

       platform              M        p         q        RMSE   MAPE_pct
0  deepseek.com    7287.399850  0.06143  0.074284   77.687406  15.429151
1   chatgpt.com  111177.737662  0.03195  0.144526  149.131812   2.504960
Saved: outputs/bass2025_parameters_scaled_millions.csv
deepseek.com {'out_csv': 'outputs/bass2025_deepseek_com.csv', 'fit_png': 'outputs/bass2025_deepseek_com_fit.png', 'cum_png': 'outputs/bass2025_deepseek_com_cumulative.png', 'preview_csv': 'outputs/bass2025_deepseek_com_preview.csv'}
chatgpt.com {'out_csv': 'outputs/bass2025_chatgpt_com.csv', 'fit_png': 'outputs/bass2025_chatgpt_com_fit.png', 'cum_png': 'outputs/bass2025_chatgpt_com_cumulative.png', 'preview_csv': 'outputs/bass2025_chatgpt_com_preview.csv'}


In [15]:
def fit_M_given_pq(t: np.ndarray, y: np.ndarray, p: float, q: float) -> float:
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)

    g = bass_F(t, p, q) - bass_F(np.maximum(t - 1.0, 0.0), p, q)
    denom = float(np.dot(g, g))
    if denom <= 0.0:
        return float(np.maximum(y.sum(), 1e-12))

    m_hat = float(np.dot(g, y) / denom)
    m_hat = max(m_hat, float(y.sum()) + 1e-9)
    return m_hat

def add_transfer_fit(source_platform: str, target_platform: str):
    src_row = fit_df.loc[fit_df["platform"] == source_platform].iloc[0]
    p_src = float(src_row["p"])
    q_src = float(src_row["q"])

    y_raw = df[target_platform].astype(float).to_numpy()
    y = y_raw / scale
    t = np.arange(1, len(y) + 1, dtype=float)

    m_tr = fit_M_given_pq(t, y, p_src, q_src)
    yhat_tr = bass_sales_by_diff(t, m_tr, p_src, q_src)

    rmse_tr = float(np.sqrt(np.mean((yhat_tr - y) ** 2)))
    mape_tr = float(np.mean(np.abs((yhat_tr - y) / np.maximum(y, 1e-12))) * 100.0)

    fit_tgt = fit_df.loc[fit_df["platform"] == target_platform].iloc[0]
    yhat_own = bass_sales_by_diff(t, float(fit_tgt["M"]), float(fit_tgt["p"]), float(fit_tgt["q"]))

    fig = plt.figure()
    plt.plot(df["month"], y, marker="o", linewidth=1, label="Observed (scaled)")
    plt.plot(df["month"], yhat_own, marker="o", linewidth=1, label=f"Own fit (RMSE={float(fit_tgt['RMSE']):.3f})")
    plt.plot(df["month"], yhat_tr, marker="o", linewidth=1, label=f"Transfer p,q from {source_platform} (RMSE={rmse_tr:.3f})")
    plt.title(f"DeepSeek: own Bass fit vs transfer (p,q) from {source_platform}")
    plt.xlabel("Month")
    plt.ylabel(f"Per-period proxy (visits / {scale:g})")
    plt.legend()
    fig.autofmt_xdate()
    out_png = out_dir / f"{prefix}_{target_platform.replace('.','_')}_transfer_from_{source_platform.replace('.','_')}_fit.png"
    fig.savefig(out_png, dpi=160, bbox_inches="tight")
    plt.close(fig)

    T = len(y)
    t_all = np.arange(1, T + forecast_months + 1, dtype=float)
    yhat_all_tr = bass_sales_by_diff(t_all, m_tr, p_src, q_src)
    Ahat_all_tr = np.cumsum(yhat_all_tr)
    months_all = pd.date_range(df["month"].iloc[0], periods=len(t_all), freq="MS")

    out = pd.DataFrame({
        "month": months_all,
        "observed_per_period": np.r_[y, [np.nan] * forecast_months],
        "fitted_per_period": bass_sales_by_diff(t_all, float(fit_tgt["M"]), float(fit_tgt["p"]), float(fit_tgt["q"])),
        "fitted_cumulative": np.cumsum(bass_sales_by_diff(t_all, float(fit_tgt["M"]), float(fit_tgt["p"]), float(fit_tgt["q"]))),
        "transfer_fitted_per_period": yhat_all_tr,
        "transfer_fitted_cumulative": Ahat_all_tr,
        "transfer_source_platform": [source_platform] * len(t_all),
        "transfer_p": [p_src] * len(t_all),
        "transfer_q": [q_src] * len(t_all),
        "transfer_M": [m_tr] * len(t_all),
    })

    out_csv = out_dir / f"{prefix}_{target_platform.replace('.','_')}_with_transfer_from_{source_platform.replace('.','_')}.csv"
    out.to_csv(out_csv, index=False)

    transfer_summary = {
        "target_platform": target_platform,
        "source_platform": source_platform,
        "M_transfer": m_tr,
        "p_transfer": p_src,
        "q_transfer": q_src,
        "RMSE_transfer": rmse_tr,
        "MAPE_pct_transfer": mape_tr,
        "overlay_png": str(out_png),
        "out_csv_with_transfer": str(out_csv),
    }
    return transfer_summary

transfer = add_transfer_fit(source_platform="chatgpt.com", target_platform="deepseek.com")
print("TRANSFER SUMMARY")
print(pd.Series(transfer))

TRANSFER SUMMARY
target_platform                                               deepseek.com
source_platform                                                chatgpt.com
M_transfer                                                     8041.181021
p_transfer                                                         0.03195
q_transfer                                                        0.144526
RMSE_transfer                                                    126.91773
MAPE_pct_transfer                                                23.415948
overlay_png              outputs/bass2025_deepseek_com_transfer_from_ch...
out_csv_with_transfer    outputs/bass2025_deepseek_com_with_transfer_fr...
dtype: object


In [17]:
from pathlib import Path
import pandas as pd
import numpy as np

tables_dir = Path("tables")
tables_dir.mkdir(parents=True, exist_ok=True)

def _fmt_table(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()

    df = df.rename(columns={
        "observed_per_period": "Observed $a_t$",
        "fitted_per_period": "Fitted $\\hat{a}_t$",
        "fitted_cumulative": "Fitted $\\hat{A}(t)$",
        "transfer_fitted_per_period": "Transfer $\\hat{a}_t$",
        "transfer_fitted_cumulative": "Transfer $\\hat{A}(t)$"
    })

    df["month"] = pd.to_datetime(df["month"]).dt.strftime("%Y-%m")

    for c in df.columns:
        if c != "month":
            df[c] = pd.to_numeric(df[c], errors="coerce").round(2)

    return df.replace({np.nan: "--"})

def write_longtable(df_in: pd.DataFrame, path: Path, caption: str, label: str):
    df = _fmt_table(df_in)

    tex = df.to_latex(
        index=False,
        escape=False,
        longtable=True,
        caption=caption,
        label=label,
        na_rep="--",
        column_format="lrrr",
        bold_rows=False
    )

    path.write_text(tex, encoding="utf-8")

chatgpt_csv = Path("outputs") / "bass2025_chatgpt_com.csv"
deepseek_csv = Path("outputs") / "bass2025_deepseek_com.csv"
deepseek_transfer_csv = Path("outputs") / "bass2025_deepseek_com_with_transfer_from_chatgpt_com.csv"

chatgpt_df = pd.read_csv(chatgpt_csv)
deepseek_df = pd.read_csv(deepseek_csv)
deepseek_tr_df = pd.read_csv(deepseek_transfer_csv)

write_longtable(
    chatgpt_df[["month","observed_per_period","fitted_per_period","fitted_cumulative"]],
    tables_dir / "chatgpt_own_longtable.tex",
    caption="OpenAI (chatgpt.com) monthly adoption proxy and Bass own-fit (millions of visits). Observed values are not available for forecast months.",
    label="tab:openai_own_values"
)

write_longtable(
    deepseek_df[["month","observed_per_period","fitted_per_period","fitted_cumulative"]],
    tables_dir / "deepseek_own_longtable.tex",
    caption="DeepSeek (deepseek.com) monthly adoption proxy and Bass own-fit (millions of visits). Observed values are not available for forecast months.",
    label="tab:deepseek_own_values"
)

write_longtable(
    deepseek_tr_df[["month","observed_per_period","transfer_fitted_per_period","transfer_fitted_cumulative"]],
    tables_dir / "deepseek_transfer_from_chatgpt_longtable.tex",
    caption="DeepSeek monthly adoption proxy and Bass transfer fit (millions of visits). Transfer model reuses $(\\hat p,\\hat q)$ from chatgpt.com and re-estimates $M$ for DeepSeek. Observed values are not available for forecast months.",
    label="tab:deepseek_transfer_values"
)

print("Wrote TeX tables to:", tables_dir)

Wrote TeX tables to: tables
